In [1]:
import os
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
import pandas as pd
import random
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import f1_score, roc_auc_score, cohen_kappa_score

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

## Dataset and Utils

In [2]:
class FF_ZINC(torch.utils.data.Dataset):
    def __init__(self, partition, num_classes=4):
        df = pd.read_excel("./data/Zinc/zincrougher.xls", header=None)
        df.iloc[(df.iloc[:, 13] >= 0.395903), 13] = 3
        df.iloc[(df.iloc[:, 13] < 0.395903) & (df.iloc[:, 13] >= -0.157400), 13] = 2
        df.iloc[(df.iloc[:, 13] < -0.157400) & (df.iloc[:, 13] >= -0.591114), 13] = 1
        df.iloc[(df.iloc[:, 13] < -0.591114), 13] = 0

        scalar = StandardScaler()
        df.iloc[:, :-1] = scalar.fit_transform(df.iloc[:, :-1])
        npdata = df.values.astype(np.float32)
        Zinc_dataset = torch.utils.data.TensorDataset(
            torch.tensor(npdata[:, :-1]), torch.tensor(npdata[:, -1]).to(torch.int64)
        )
        idx = [i for i in range(len(Zinc_dataset))]
        random.shuffle(idx)
        print(idx)
        if partition == "train":
            self.Zinc = torch.utils.data.Subset(Zinc_dataset, idx[:3153])
        elif partition == "val":
            self.Zinc = torch.utils.data.Subset(Zinc_dataset, idx[3153:4053])
        elif partition == "test":
            self.Zinc = torch.utils.data.Subset(Zinc_dataset, idx[4053:])

        self.num_classes = num_classes
        self.uniform_label = torch.ones(self.num_classes) / self.num_classes

    def __getitem__(self, index):
        pos_sample, neg_sample, neutral_sample, original_sample, class_label = (
            self._generate_sample(index)
        )

        inputs = {
            "pos_sample": pos_sample,
            "neg_sample": neg_sample,
            "natrual_sample": neutral_sample,
            "original_sample": original_sample,
        }
        labels = {"class_labels": class_label}
        return inputs, labels

    def __len__(self):
        return len(self.Zinc)

    def _get_pos_sample(self, sample, class_label):
        one_hot_label = torch.nn.functional.one_hot(
            torch.tensor(class_label), num_classes=self.num_classes
        )
        pos_sample = sample.clone()
        pos_sample = torch.cat(
            [one_hot_label.unsqueeze(0), pos_sample.unsqueeze(0)],
            dim=1,
        )
        # pos_sample[:, 0, : self.num_classes] = one_hot_label
        return pos_sample

    def _get_neg_sample(self, sample, class_label):
        # Create randomly sampled one-hot label.
        classes = list(range(self.num_classes))
        classes.remove(class_label)  # Remove true label from possible choices.
        wrong_class_label = np.random.choice(classes)
        one_hot_label = torch.nn.functional.one_hot(
            torch.tensor(wrong_class_label), num_classes=self.num_classes
        )
        neg_sample = sample.clone()
        # neg_sample = torch.cat(
        #     [one_hot_label.unsqueeze(0), neg_sample.reshape(neg_sample.shape[0], -1)],
        #     dim=1,
        # )
        neg_sample = torch.cat(
            [one_hot_label.unsqueeze(0), neg_sample.unsqueeze(0)],
            dim=1,
        )
        # neg_sample[:, 0, : self.num_classes] = one_hot_label
        return neg_sample

    def _get_neutral_sample(self, z):
        # z = torch.cat(
        #     [self.uniform_label.unsqueeze(0), z.reshape(z.shape[0], -1)], dim=1
        # )
        z = torch.cat([self.uniform_label.unsqueeze(0), z.unsqueeze(0)], dim=1)
        # z[:, 0, : self.num_classes] = self.uniform_label
        return z

    def _get_original_sample(self, z):
        return z

    def _generate_sample(self, index):
        # Get MNIST sample.
        sample, class_label = self.Zinc[index]
        pos_sample = self._get_pos_sample(sample, class_label)
        neg_sample = self._get_neg_sample(sample, class_label)
        neutral_sample = self._get_neutral_sample(sample)
        original_sample = self._get_original_sample(sample)
        return pos_sample, neg_sample, neutral_sample, original_sample, class_label

In [3]:
train_set = FF_ZINC("train")
val_set = FF_ZINC("val")
test_set = FF_ZINC("test")
trn_loader = torch.utils.data.DataLoader(
    train_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
tst_loader = torch.utils.data.DataLoader(
    test_set,
    128,
    drop_last=False,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)

[1253, 1780, 551, 2279, 4065, 2972, 4099, 487, 3895, 2192, 722, 583, 2778, 4482, 441, 2554, 3216, 1334, 3761, 2151, 2797, 4502, 4384, 2726, 4075, 2268, 1147, 1739, 4236, 4022, 1325, 3996, 48, 232, 2326, 1539, 166, 457, 3584, 4346, 733, 3278, 2313, 911, 221, 236, 3266, 478, 4293, 1281, 3995, 1156, 71, 730, 3416, 431, 3885, 3568, 1538, 1215, 3821, 3963, 4198, 4110, 2050, 4438, 1807, 2307, 1625, 1752, 3289, 619, 2242, 694, 4357, 900, 1137, 4286, 4218, 1759, 3138, 1446, 4055, 805, 800, 1877, 3985, 2903, 857, 2969, 2291, 3473, 775, 723, 449, 2782, 841, 544, 4280, 352, 3304, 2944, 2945, 2495, 1565, 2054, 742, 692, 950, 3646, 4285, 2667, 1977, 2133, 1797, 1727, 2601, 337, 557, 3426, 2443, 2015, 704, 2143, 672, 3445, 821, 2532, 4127, 1705, 1929, 545, 98, 3197, 2501, 3647, 73, 4014, 3018, 4428, 2315, 790, 929, 1010, 826, 148, 4097, 3724, 1187, 3318, 297, 1987, 1832, 2374, 194, 2630, 3002, 2566, 435, 753, 4267, 689, 3204, 2857, 3570, 2021, 3514, 3780, 3948, 54, 42, 3662, 3358, 2009, 2073, 3400, 

In [4]:
def ts_append(a, b):
    """List like 'Append' tool for tensor datatype
    ---
    Parameters:
        a, b: append a with b
    """
    if a is None:
        return b
    else:
        return torch.cat([a, b], dim=0)

## Validation

In [5]:
def valid(net, valid_data):
    yAll = None
    outAll = None
    outProb = None
    outembedding = None

    with torch.no_grad():
        for data in valid_data:
            x, y = data
            x = x["original_sample"].cuda()
            y = y["class_labels"].cuda()
            x = x.reshape(x.shape[0], -1)

            output = net(x)
            outembedding = ts_append(outembedding, output)
            outAll = ts_append(outAll, output.argmax(1))
            outProb = ts_append(outProb, output)
            yAll = ts_append(yAll, y)

    acc = outAll.eq(yAll).float().mean().item()
    f1_mac = f1_score(outAll.cpu().numpy(), yAll.cpu().numpy(), average="macro")
    f1_mic = cohen_kappa_score(outAll.cpu().numpy(), yAll.cpu().numpy())
    auc_s = roc_auc_score(yAll.cpu().numpy(), outProb.cpu().numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s, outembedding, yAll)

In [6]:
def valid_no_model(output, y):
    acc = sum(output == y) / (output.shape[0])
    f1_mac = f1_score(output, y, average="macro")
    f1_mic = cohen_kappa_score(output, y)
    auc_s = roc_auc_score(output, F.one_hot(torch.tensor(y)).numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s)

## Model MLP

In [7]:
class MLPNet(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(MLPNet, self).__init__()
        self.model = nn.ModuleList(
            [
                nn.Linear(in_features, hidden_features),
                nn.Linear(hidden_features, out_features),
            ]
        )
        self.relu = nn.ReLU()

    def forward(self, input):
        hidden = self.relu(self.model[0](input))
        return F.softmax(self.model[1](hidden), dim=1)

## Train MLP

In [8]:
lr = 1e-3
wd = 1e-5
epoch = 500

model = MLPNet(13, 1000, 4).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=lr)

pbar = tqdm(total=epoch)
for i in range(epoch):
    pbar.set_description_str(f"Epoch: {i}/{epoch}")
    total_loss = 0
    for inputs, labels in trn_loader:
        inputs = inputs["original_sample"].cuda()
        labels = labels["class_labels"].cuda()
        opt.zero_grad()
        inputs = inputs.reshape(inputs.shape[0], -1)

        output = model(inputs)
        loss = F.nll_loss(output, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    trn_acc, _, _, _, _, _ = valid(model, trn_loader)
    val_acc, _, _, _, _, _ = valid(model, val_loader)
    if i % 1 == 0:
        test_acc, _, _, _, _, _ = valid(model, tst_loader)

    total_loss = total_loss / len(trn_loader)
    pbar.set_postfix(
        loss=total_loss, trn_acc=trn_acc, val_acc=val_acc, test_acc=test_acc
    )
    pbar.update(1)

with torch.no_grad():
    test_acc, test_f1_mac, test_f1_mic, test_auc, BP_FNN_output, BP_FNN_target = valid(
        model, tst_loader
    )
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "CK:",
    test_f1_mic,
    "auc",
    test_auc,
)
pbar.close()

Epoch: 0/500:   0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_2805460/593617411.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/tmp/ipykernel_2805460/593617411.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/tmp/ipykernel_2805460/593617411.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/tmp/ipykernel_2805460/593617411.py:47: UserWarning: To copy cons

KeyboardInterrupt: 